# Breast Cancer Veri Seti ile Başlangıç

Bu notebook, ödevin ilk aşaması için başlangıç seviyesinde hazırlandı. Amacımız veri setini yüklemek, tabloyu
görmek, kolonları incelemek ve temel yapısını anlamak.


## 1. Veri Setinin Yüklenmesi

Bu bölümde veri setini önce adım adım inceleyip sonra bölüm sonunda tek parça çalışan kod ile toparlıyoruz.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer


In [ ]:
dataset = load_breast_cancer()

print(type(dataset))
print(dataset.keys())

print(dataset.data[:5])

İlk adımda veri setinin yapısını ham hâliyle gördük. Böylece elimizde nasıl bir nesne olduğunu ve veri seti
içinde hangi alanların bulunduğunu anlamış olduk.


## 1.1 DataFrame'e Dönüştürme

In [ ]:
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')

## 1.2 İlk Gözlemler ve Boyut Bilgisi

In [ ]:
print('Satır ve sütun sayısı:', X.shape)
print('Hedef değişken uzunluğu:', y.shape)

In [ ]:
X.head()

In [ ]:
y.head()

## 1.3 Özellikleri ve Hedefi Birlikte Görmek

In [ ]:
df = X.copy()
df['target'] = y

In [ ]:
print(df.head())

In [ ]:
print(df['target'].value_counts())

## 1.4 Bölümün Toplu Çalışan Kodu

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')

print('Satır ve sütun sayısı:', X.shape)
print('Hedef değişken uzunluğu:', y.shape)

df = X.copy()
df['target'] = y

print(df.head())
print(df['target'].value_counts())

## 1. Bölüm Sonu Yorumu

Breast Cancer veri seti scikit-learn kütüphanesinden yüklenmiştir. Veri seti özellikler (`X`) ve hedef
değişken (`y`) olarak ayrılmış, pandas DataFrame yapısına dönüştürülmüş ve ilk satırlar incelenmiştir. Bu
aşamada veri setinin genel yapısı ve değişkenleri tanınmıştır.


# 2. Veri Seti Kalite Kontrolleri

Bu bölümde veri kalitesini önce adım adım inceleyip sonra bölüm sonunda kısa bir toplu kontrol kodu ile toparlıyoruz.

## 2.1 Eksik Değer Analizi

In [ ]:
missing_values = X.isnull().sum()
missing_values

In [ ]:
print('Toplam eksik değer sayısı:', X.isnull().sum().sum())

Bu aşamada sütun sütun eksik değer sayılarını gördük. Eğer tüm sütunlarda sonuç `0` ise veri setinde eksik
veri bulunmadığını söyleyebiliriz.


## 2.2 Veri Tiplerini İnceleme

In [ ]:
X.dtypes

In [ ]:
X.dtypes.value_counts()

Bu çıktı bize veri setindeki sütunların hangi veri tipinde olduğunu gösterir. Tüm değişkenlerin sayısal
olması, sonraki analiz adımlarını daha rahat uygulamamızı sağlar.


## 2.3 Aykırı Değer Analizi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(18, 10))
sns.boxplot(data=X)
plt.xticks(rotation=90)
plt.title('Tüm Özellikler İçin Boxplot Grafiği')
plt.show()

Boxplot grafikleri bazı sütunlarda aykırı değer olabilecek gözlemler bulunduğunu göstermektedir. Özellikle
kutu grafiğinin dışında kalan noktalar dikkat çekmektedir. Bu değerler bazı modelleri etkileyebileceği için
sonraki adımlarda ölçeklendirme ve model seçimi önemli olacaktır.


## 2.4 Bölümün Toplu Çalışan Kodu

In [ ]:
missing_values = X.isnull().sum()
print(missing_values)

print('Toplam eksik değer sayısı:', X.isnull().sum().sum())

print(X.dtypes)
print(X.dtypes.value_counts())

## 2. Bölüm Sonu Yorumu

Veri setinde yapılan kalite kontrolleri sonucunda hiçbir sütunda eksik gözlem bulunmadığı görülmüştür. Ayrıca
tüm değişkenlerin sayısal ve `float64` tipinde olduğu belirlenmiştir. Boxplot incelemesi ise bazı
değişkenlerde aykırı değer olabilecek gözlemler bulunduğunu göstermiştir. Bu durum, veri setinin genel olarak
analize uygun olduğunu; ancak ölçeklendirme ve modelleme aşamalarında aykırı değer etkisinin dikkate alınması
gerektiğini göstermektedir.


# 3. Keşifsel Veri Analizi (EDA)

Bu bölümde, modeli kurmadan önce veriyi daha iyi tanımak için temel istatistiklere ve değişkenler arası ilişkilere bakıyorum.

## 3.1 Temel İstatistikler

Önce sayısal değişkenlerin özetini (ortalama, standart sapma, çeyreklikler vb.) çıkarıyorum.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# df, 1. bölümde oluşturulmuştu (X + target).
# Burada EDA için hızlı bir özet tablo alıyorum.
istatistik_ozet = df.drop(columns=['target']).describe().T
istatistik_ozet.head()


In [ ]:
# İstersem tüm tabloyu da görüntüleyebilirim:
istatistik_ozet


In [ ]:
# Target'a göre bazı özelliklerin ortalaması nasıl değişiyor?
df.groupby('target').mean(numeric_only=True).iloc[:, :8]  # ilk 8 özelliği örnek olarak gösteriyorum


## 3.2 Korelasyon Matrisi ve Heatmap

Korelasyon, iki değişkenin birlikte nasıl değiştiğini gösterir. Çok yüksek korelasyonlar bazı modellerde
(özellikle doğrusal) benzer bilgiyi tekrar taşıyor olabilir.


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Tüm özellikler için korelasyon matrisi
korelasyon = X.corr()

# Heatmap çok kalabalık olmasın diye, hedef değişkenle en ilişkili ilk 12 özelliği seçiyorum
hedef_korelasyon = X.corrwith(y).abs().sort_values(ascending=False)
secili_ozellikler = hedef_korelasyon.index[:12]
korelasyon_secili = korelasyon.loc[secili_ozellikler, secili_ozellikler]

plt.figure(figsize=(10, 8))
sns.heatmap(korelasyon_secili, cmap='coolwarm', center=0, annot=False)
plt.title('Korelasyon Heatmap (Seçili 12 Özellik)')
plt.tight_layout()
plt.savefig('../figures/03_korelasyon_heatmap.svg')
plt.show()


In [ ]:
# En yüksek mutlak korelasyona sahip ilk 3 değişken çifti
korelasyon_abs = korelasyon.abs()
ust_ucgen = korelasyon_abs.where(np.triu(np.ones(korelasyon_abs.shape), k=1).astype(bool))
en_yuksek = (
    ust_ucgen.stack()
    .sort_values(ascending=False)
    .head(3)
)
en_yuksek


## 3. Bölüm Sonu Yorumu

Bu adımda verinin genel dağılımını ve değişkenlerin birbiriyle ilişkisini gördüm. Korelasyon matrisi özellikle
çok benzer bilgi taşıyan özellikleri fark etmeme yardımcı oldu.


# 4. Veri Bölünmesi ve Ölçeklendirme

Bu aşamada veriyi `train / validation / test` olarak ayırıyorum. Daha sonra ölçeklendirmeyi sadece `train` 
verisine göre öğrenip (`fit`), diğerlerine aynı dönüşümü uyguluyorum (`transform`). Böylece veri sızıntısı 
(data leakage) riskini azaltmış olurum.


## 4.1 Train / Validation / Test Bölünmesi (70 / 10 / 20)

Ödevde istenen oranları korumak için önce `%20` test ayırıp, kalan `%80` kısmı kendi içinde `%70` train ve 
%10 validation olacak şekilde tekrar bölüyorum. `stratify=y` ile sınıf oranlarını yaklaşık sabit tutuyorum.


In [ ]:
from sklearn.model_selection import train_test_split

# 1) Önce test setini ayır (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2) Kalan 80% içinden validation ve test'i eşit böl (10% + 10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val  :', X_val.shape,   'y_val  :', y_val.shape)
print('X_test :', X_test.shape,  'y_test :', y_test.shape)


In [ ]:
# Sınıf dağılımları yaklaşık korunuyor mu?
print('train target dağılımı:\n', y_train.value_counts(normalize=True))
print('val   target dağılımı:\n', y_val.value_counts(normalize=True))
print('test  target dağılımı:\n', y_test.value_counts(normalize=True))


## 4.2 Ölçeklendirme (StandardScaler)

Özelliklerin ölçekleri farklı olduğu için `StandardScaler` kullanıyorum. En önemli nokta: scaler'ı sadece 
train verisiyle `fit` etmek.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Kolon isimlerini kaybetmemek için tekrar DataFrame'e çeviriyorum
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns, index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

X_train_scaled.head()


## 4. Bölüm Sonu Yorumu

Bu adımda veriyi 70/10/20 oranında böldüm ve ölçeklendirmeyi sadece train verisine göre öğrenerek uyguladım. 
Böylece sonraki PCA/LDA ve modelleme adımlarına daha sağlıklı bir şekilde geçebilirim.
